# Advanced Python: acute-to-12-month recovery prediction

This optional notebook uses a separate synthetic cohort. It defines predictors available at
an acute assessment and estimates a synthetic 12-month language outcome.

The exercise evaluates a prediction workflow. It does not provide external validation,
clinical utility, calibration, or evidence from real participants.

For each task, copy the prompt into an AI assistant, paste the returned code into the empty
cell, run it, and compare the result with the checkpoint.

## Setup

Run this cell first.

In [ ]:
import os, subprocess, sys

REPO = "mjmarte/ai-coding-workshop"
if os.path.isdir(".git"):
    result = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif os.path.isdir("ai-coding-workshop"):
    os.chdir("ai-coding-workshop")
    result = subprocess.run(["git", "pull", "--ff-only", "-q"])
elif not os.path.exists("data"):
    result = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git"])
    if result.returncode == 0:
        os.chdir("ai-coding-workshop")
else:
    result = subprocess.run(["git", "status"], capture_output=True)
if result.returncode != 0:
    raise SystemExit("Could not download or update the workshop files. Check the GitHub link.")

missing = [path for path in ("data/recovery_prediction.csv",) if not os.path.exists(path)]
if missing:
    raise SystemExit(f"Setup incomplete. Missing: {missing}")
print("Ready.")

---
## Task A1 - Define the prediction question before fitting a model

Goal: identify the outcome, specify predictors available at the acute assessment,
and confirm one row per participant.

The prediction question requires a time boundary. The 12-month outcome and later measurements
are excluded from the acute predictor matrix.

Copy this prompt into your AI assistant:

```
I am working in Python with `data/recovery_prediction.csv`, a synthetic cohort with one
row per acute-stroke participant. My outcome is `outcome_wab_aq_12m`, a 12-month WAB-AQ.

The following variables are available at the acute assessment: age, sex, education_years,
acute_wab_aq, acute_discourse_score, lesion_volume_ml, cortical_lesion_pct,
dorsal_disconnection_pct, and ventral_disconnection_pct.

Write pandas code that loads the file into `recovery`, verifies that participant_id is unique,
prints the sample size and missing-value count per column, and creates two named predictor
lists:
- `clinical_features`: age, education_years, acute_wab_aq, acute_discourse_score
- `multimodal_features`: every clinical feature plus lesion_volume_ml, cortical_lesion_pct,
  dorsal_disconnection_pct, and ventral_disconnection_pct

Do not put participant_id, sex, or outcome_wab_aq_12m into either predictor list. Give only
code with short comments.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> Checkpoint: Expected: 90 participants, one row per participant, and no outcome or identifier in either predictor list.

---
## Task A2 - Compare baseline and multimodal models without leakage

Goal: compare a clinical baseline model with a model that additionally includes acute
lesion and disconnection measures.

Imputation, scaling, and fitting occur within each resampling fold so that held-out records do
not influence the training procedure.

Copy this prompt into your AI assistant:

```
Using `recovery`, `outcome`, `clinical_features`, and `multimodal_features`, compare two
Ridge-regression pipelines: a clinical model and a clinical-plus-imaging model.

Use 5-fold repeated cross-validation with 20 repeats and random_state=202607. Each pipeline
must impute missing values, standardize predictors, and fit RidgeCV with alphas
[0.01, 0.1, 1, 10, 100]. Keep all preprocessing inside the Pipeline.

For each feature set, report the mean and SD of cross-validated mean absolute error and R-squared.
Return a two-row DataFrame named `cv_summary`. Do not fit on the full dataset before evaluating.
Give only code with short comments.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> Checkpoint: In the supplied synthetic data, the multimodal model has lower resampled MAE. This is not evidence of clinical utility.

---
## Task A3 - Inspect out-of-fold predictions and error distribution

Goal: generate held-out predictions and inspect their errors alongside the resampled summary.

Copy this prompt into your AI assistant:

```
Using the multimodal feature set, generate one out-of-fold prediction per participant with
5-fold shuffled KFold cross-validation and random_state=202607. Keep imputation, scaling, and
RidgeCV inside a Pipeline. Store predictions in `oof_pred`.

Make a two-panel matplotlib figure: observed versus predicted 12-month WAB-AQ with an identity
line, and absolute prediction error by sex with individual points. Print overall MAE and MAE by
sex with the number of participants per group.

Treat the sex comparison as a descriptive error audit, not evidence of a subgroup difference.
Give only code with short comments.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> Checkpoint: Each point in the first panel was predicted without fitting that participant. The displayed subgroup errors are descriptive.

---
## Task A4 - Fact-check an AI-written prediction summary

Goal: constrain an AI-written summary to the resampling output and the scope of a
synthetic development exercise.

Cross-validation estimates performance under the specified resampling procedure. It does not
establish clinical utility, causal contribution, external validity, calibration, or deployment readiness.

Copy this prompt into your AI assistant:

```
Here is my cross-validation summary table from a synthetic development exercise:

[PASTE cv_summary HERE]

Write two Results sentences. Report only numbers in the table. State which model had lower
resampled MAE. Do not call the model clinically useful, externally validated, calibrated,
causal, or ready for deployment.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> Checkpoint: A correct response reports only the resampled metrics shown in `cv_summary` and makes no clinical, causal, or external-validation claim.

---
## Task A5 - Use an agent for a bounded reproducibility audit

Goal: ask an agent to inspect the project without authority to alter data or
analysis files.

An agent can inventory files and identify consistency questions. It does not determine the
predictor set or the validity of a clinical claim.

Copy this prompt into your AI assistant:

```
I am in the `ai-coding-workshop` repository. Perform a READ-ONLY reproducibility audit of
the advanced recovery-prediction exercise. Do not edit, create, delete, or run any file.

Return a compact table with: (1) raw or generated input files, (2) the script that produces each
input, (3) the notebook that consumes it, (4) model outcome and predictors, and (5) one leakage
or interpretation risk at each stage. Flag any claim in the materials that exceeds what a
synthetic development exercise can establish. Cite file paths and line numbers where possible.
```

In [ ]:
# paste the AI's code here, then press Shift+Enter to run it

> Checkpoint: A useful agent response cites the generator, the CSV, this notebook, and the resampling code. It does not make clinical claims or edit the project.